### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

### Load the Dataset

In [2]:
df = pd.read_csv('../data/processed/rainfall_model_ready.csv')

In [3]:
df = df.sort_values(
    ['YEAR', 'SUBDIVISION']
).reset_index(drop=True)

In [4]:
print("Shape:", df.shape)
df.head()

Shape: (3874, 27)


,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,...,Jun-Sep,Oct-Dec,annual_lag1,annual_lag2,annual_lag3,rolling_mean_3,annual_lag4,annual_lag5,rolling_mean_5,target
0,ANDAMAN & NICOBAR ISLANDS,1906,36.6,0.0,0.0,0.0,556.1,733.3,247.7,320.5,...,1465.8,475.9,2566.7,3079.6,2957.4,2867.900000,3520.7,3373.2,3099.52,3347.9
1,ASSAM & MEGHALAYA,1906,9.7,59.2,88.4,263.5,271.3,388.4,541.8,585.9,...,1780.3,190.8,2790.7,2664.2,2544.6,2666.500000,2952.1,2498.7,2690.06,2521.3
2,BIHAR,1906,14.4,56.9,8.6,0.9,44.7,191.3,366.2,430.1,...,1106.3,33.3,1498.6,1269.7,1016.4,1261.566667,1206.2,952.4,1188.66,1126.4
3,CHHATTISGARH,1906,25.0,91.0,52.5,0.0,4.1,210.1,445.2,258.3,...,1155.9,66.1,1250.9,1492.4,1331.1,1358.133333,923.9,1386.9,1277.04,1308.8
4,COASTAL ANDHRA PRADESH,1906,3.9,23.5,9.9,2.3,11.0,252.6,155.8,241.1,...,776.4,194.4,795.0,860.1,1316.1,990.400000,1063.5,993.7,1005.68,817.9


### Define Features

In [5]:
feature_cols = [
    'annual_lag1',
    'annual_lag2',
    'annual_lag3',
    'annual_lag4',
    'annual_lag5',
    'rolling_mean_3',
    'rolling_mean_5'
]

target_col = 'target'

### Train Test Split

In [6]:
train_df = df[
    df['YEAR'] <= 2000
].copy()

validation_df = df[
    (df['YEAR'] > 2000) &
    (df['YEAR'] <= 2008)
].copy()

final_holdout_df = df[
    (df['YEAR'] > 2008) &
    (df['YEAR'] <= 2014)
].copy()

In [7]:
print("Training:", train_df['YEAR'].min(), "to", train_df['YEAR'].max())
print("Validation:", validation_df['YEAR'].min(), "to", validation_df['YEAR'].max())
print("Final holdout:", final_holdout_df['YEAR'].min(), "to", final_holdout_df['YEAR'].max())

print("\nRows:")
print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Final holdout:", len(final_holdout_df))

Training: 1906 to 2000
Validation: 2001 to 2008
Final holdout: 2009 to 2014

Rows:
Training: 3370
Validation: 288
Final holdout: 216


### Create X and Y

In [8]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_validation = validation_df[feature_cols]
y_validation = validation_df[target_col]

X_holdout = final_holdout_df[feature_cols]
y_holdout = final_holdout_df[target_col]

In [9]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_holdout:", X_holdout.shape)
print("y_holdout:", y_holdout.shape)

X_train: (3370, 7)
y_train: (3370,)
X_validation: (288, 7)
y_validation: (288,)
X_holdout: (216, 7)
y_holdout: (216,)


### Train Final Linear Regression Model

In [10]:
final_model = LinearRegression()

final_model.fit(
    X_train,
    y_train
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](7,)","[0.22,0.14,0.05,...,0.17,0.14,0.14]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](7,)","['annual_lag1','annual_lag2','annual_lag3',...,'annual_lag5', 'rolling_mean_3','rolling_mean_5']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,42.81
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)


### Prediction and Evaluation

In [11]:
validation_pred = final_model.predict(
    X_validation
)

validation_mae = mean_absolute_error(
    y_validation,
    validation_pred
)

validation_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        validation_pred
    )
)

validation_r2 = r2_score(
    y_validation,
    validation_pred
)

print("Validation MAE:", validation_mae)
print("Validation RMSE:", validation_rmse)
print("Validation R²:", validation_r2)

Validation MAE: 224.26198826661027
Validation RMSE: 291.2273444707588
Validation R²: 0.8775979047381097


### Final Holdout Evaluation

In [12]:
holdout_pred = final_model.predict(
    X_holdout
)

In [13]:
holdout_mae = mean_absolute_error(
    y_holdout,
    holdout_pred
)

holdout_rmse = np.sqrt(
    mean_squared_error(
        y_holdout,
        holdout_pred
    )
)

holdout_r2 = r2_score(
    y_holdout,
    holdout_pred
)

print("Final Holdout MAE:", holdout_mae)
print("Final Holdout RMSE:", holdout_rmse)
print("Final Holdout R²:", holdout_r2)

Final Holdout MAE: 211.83981251906286
Final Holdout RMSE: 289.06099919324186
Final Holdout R²: 0.8910400567984633


In [14]:
import joblib

joblib.dump(
    final_model,
    '../models/linear_regression_rainfall.joblib'
)

print("Model saved successfully.")

Model saved successfully.


In [15]:
import os

model_path = '../models/linear_regression_rainfall.joblib'

print("Model exists:", os.path.exists(model_path))

Model exists: True


In [16]:
import sys
print(sys.executable)

C:\Users\HP\OneDrive\Desktop\Rainfall_Prediction\.venv\Scripts\python.exe


In [17]:
import sklearn
print("sklearn:", sklearn.__version__)

sklearn: 1.9.1
